# University Waste Management & Agricultural Field Monitoring System
### HSV Green-Masking + YOLOv8 — Colab GPU Pipeline

This notebook runs the full pipeline end-to-end on a Colab GPU runtime:

1. Environment setup (GPU check, Drive mount, dependencies)
2. Dataset download (Kaggle) + sanity inspection
3. Dataset assembly: single-class merge, train/val/test split
4. HSV green-masking preprocessing (visual pipeline check)
5. Baseline (unmasked) vs HSV-masked training — the core ablation
6. Metrics comparison (mAP50, mAP50-95, precision, recall)
7. Inference on original RGB images with green-ratio post-filter
8. Persist everything to Google Drive

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better).

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/university-waste-management'
import os
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Clone the project repo (containing `src/`) into the Colab runtime. Replace `REPO_URL`
if you've pushed this project to GitHub — otherwise upload the `src/` folder manually via the
Colab file browser into `/content/University-Waste-Management/`.

In [1]:
REPO_URL = 'https://github.com/ahmedtayel714/University-Waste-Management.git'
CODE_DIR = '/content/University-Waste-Management'

if REPO_URL:
    !git clone -q {REPO_URL} {CODE_DIR}
else:
    os.makedirs(CODE_DIR, exist_ok=True)
    print('REPO_URL not set — upload/sync src/ into', CODE_DIR, 'manually before continuing.')

import sys
sys.path.insert(0, CODE_DIR)

NameError: name 'os' is not defined

### Resuming a previous session?

If you already trained baseline/masked/waste/leaf models in an earlier
session, run the cell below to recreate the path variables those training
cells would have set — **without re-running training**. Skip it on a
genuinely first run (the variables get set naturally as you go through the
notebook the first time).

In [ ]:
RESUMING_PREVIOUS_SESSION = False  # flip to True if models already exist on Drive

if RESUMING_PREVIOUS_SESSION:
    from pathlib import Path
    from src.preprocessing.hsv_mask import MaskConfig

    RUNS_DIR = f'{PROJECT_ROOT}/runs'
    RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
    BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
    MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
    WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'

    baseline_run_dir = f'{RUNS_DIR}/baseline'
    masked_run_dir = f'{RUNS_DIR}/masked'
    masked_weights = f'{RUNS_DIR}/masked/weights/best.pt'
    waste_weights = f'{RUNS_DIR}/waste/weights/best.pt'
    leaf_weights = f'{RUNS_DIR}/leaf/weights/best.pt'

    mask_config = MaskConfig()

    for name, path in [('masked_weights', masked_weights), ('waste_weights', waste_weights), ('leaf_weights', leaf_weights)]:
        exists = Path(path).exists()
        print(f'{name}: {"found" if exists else "NOT FOUND"} at {path}')
else:
    print('Fresh run — variables will be set naturally as you go.')

In [ ]:
!pip install -q ultralytics opencv-python-headless kaggle pyyaml tqdm seaborn

### All project imports (run this once per session)

Every `src/` function used anywhere in this notebook, imported up front —
so skipping a training/generation cell you don't need (because the model
or dataset already exists on Drive) never leaves you with a `NameError`
for something that cell happened to import as a side effect. Individual
cells further down still show their own imports too, as documentation of
what each step actually needs — re-importing an already-imported name is
a no-op, so there's no harm running both.

In [ ]:
from pathlib import Path
import cv2
import json

from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline, green_mask
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, find_class_names,
    split_dataset, write_data_yaml, build_masked_variant,
)
from src.training.train import TrainConfig, train, validate
from src.evaluation.metrics import (
    load_results_csv, summarize_final_metrics, plot_loss_curves, plot_map_curves, compare_runs,
)
from src.evaluation.report import plot_pipeline_grid
from src.inference.predict import predict_image, predict_and_save, draw_detections, Detection
from src.inference.combined_predict import predict_combined, predict_combined_and_save
from src.analysis.leaf_counter import count_leaves, count_leaves_in_regions

# Track B (only needed once you reach Part 2 — harmless to import now too)
from src.synthetic.cutout_extractor import extract_cutouts_from_yolo_seg_dataset, cutout_from_bbox
from src.synthetic.background_harvester import harvest_from_green_dataset, harvest_from_boxed_dataset
from src.synthetic.compositor import compose_scene
from src.synthetic.generate_dataset import generate_synthetic_dataset
from src.inference.track import predict_image_response, track_video_to_json, track_video

print('All src/ modules imported.')

## 2. Dataset download (Kaggle)

Upload your `kaggle.json` API token when prompted (Kaggle account → Settings → Create New API Token).

In [ ]:
from google.colab import files
import os

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
    uploaded = files.upload()  # select kaggle.json
    for fname in uploaded:
        os.rename(fname, f'{kaggle_dir}/kaggle.json')
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials ready.')

In [ ]:
RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

!kaggle datasets download -d ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes -p {RAW_DATA_DIR} --unzip
!find {RAW_DATA_DIR} -maxdepth 3 | head -30

## 3. Dataset assembly

Discover image/label pairs, **inspect the class distribution before trusting the 0/1 class-id convention**, merge crop+weed into a single `green_vegetation` class, and split train/val/test.

In [ ]:
from pathlib import Path
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, split_dataset, write_data_yaml
)

pairs = discover_pairs(Path(RAW_DATA_DIR))
print(f'Found {len(pairs)} image/label pairs')
print('Class distribution (verify before merging!):', inspect_class_distribution(pairs))

In [ ]:
BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
split_dataset(pairs, BASELINE_DIR, train=0.7, val=0.2, test=0.1, seed=42, merge_to_single_class=True)

baseline_yaml = write_data_yaml(BASELINE_DIR / 'data.yaml', BASELINE_DIR, names=['green_vegetation'])
print('Baseline data.yaml:', baseline_yaml)

## 4. HSV green-masking — visual sanity check

Inspect the mask and both masking strategies (hard black-out vs soft desaturation) on a few sample images before committing to a full-dataset pass.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline

sample_paths = list((BASELINE_DIR / 'images' / 'train').iterdir())[:3]
mask_config = MaskConfig()  # tune DEFAULT_LOWER_GREEN/UPPER_GREEN in hsv_mask.py if needed

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(16, 4 * len(sample_paths)))
for row, img_path in enumerate(sample_paths):
    image = cv2.imread(str(img_path))
    mask, hard, soft = visualize_pipeline(image, mask_config)
    for col, (title, img) in enumerate([
        ('original', image), ('mask', mask), ('hard-masked', hard), ('soft-masked', soft)
    ]):
        ax = axes[row, col] if len(sample_paths) > 1 else axes[col]
        cmap = 'gray' if img.ndim == 2 else None
        disp = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
fig.tight_layout()
plt.show()

**If the mask misses vegetation or catches soil/straw**, tune `DEFAULT_LOWER_GREEN` /
`DEFAULT_UPPER_GREEN` in `src/preprocessing/hsv_mask.py` (or pass a custom `MaskConfig` above)
and re-run this cell before generating the full masked dataset.

In [ ]:
from src.preprocessing.dataset_prep import build_masked_variant

MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
build_masked_variant(BASELINE_DIR, MASKED_DIR, mode='soft', config=mask_config)
masked_yaml = write_data_yaml(MASKED_DIR / 'data.yaml', MASKED_DIR, names=['green_vegetation'])
print('Masked data.yaml:', masked_yaml)

## 5. Train — baseline vs HSV-masked ablation

Same architecture, same hyperparameters, only the preprocessing differs. This is the core evidence for the project's thesis.

In [ ]:
from src.training.train import TrainConfig, train

RUNS_DIR = f'{PROJECT_ROOT}/runs'

baseline_cfg = TrainConfig(
    data_yaml=str(baseline_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='baseline',
)
baseline_model, baseline_results = train(baseline_cfg)

In [ ]:
masked_cfg = TrainConfig(
    data_yaml=str(masked_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='masked',
)
masked_model, masked_results = train(masked_cfg)

## 6. Compare metrics

In [ ]:
from src.evaluation.metrics import load_results_csv, plot_loss_curves, plot_map_curves, compare_runs

baseline_run_dir = f'{RUNS_DIR}/baseline'
masked_run_dir = f'{RUNS_DIR}/masked'

baseline_df = load_results_csv(baseline_run_dir)
masked_df = load_results_csv(masked_run_dir)

plot_loss_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_loss.png', 'Baseline — Loss')
plot_loss_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_loss.png', 'HSV-Masked — Loss')
plot_map_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_map.png', 'Baseline — mAP')
plot_map_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_map.png', 'HSV-Masked — mAP')

summary = compare_runs(baseline_run_dir, masked_run_dir, f'{PROJECT_ROOT}/reports/comparison.png')
summary

## 7. Inference on original RGB images

The masked model still runs on **unmasked** images at inference time — masking is a training-time noise filter only. The optional green-ratio post-filter drops boxes that don't actually contain green pixels.

In [ ]:
from src.inference.predict import predict_and_save

test_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]
masked_weights = f'{masked_run_dir}/weights/best.pt'

for img_path in test_images:
    out_path = f'{PROJECT_ROOT}/reports/inference/{img_path.stem}_pred.jpg'
    predict_and_save(
        masked_weights, img_path, out_path,
        conf=0.25, green_ratio_threshold=0.15, mask_config=mask_config,
    )
print('Saved annotated predictions to', f'{PROJECT_ROOT}/reports/inference/')

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

preds = sorted(Path(f'{PROJECT_ROOT}/reports/inference').glob('*_pred.jpg'))[:5]
fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
for ax, p in zip(axes, preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 7b. Combined pipeline figure — before → mask → detection → leaf count

The report/competition-ready artifact: one figure per sample showing
**Original → Green Mask → Masked (training view) → Final Detection → Leaf
Count**, side by side, so the whole preprocessing + counting story is
visible at a glance. Leaf counting is a classical watershed split on the
mask (see `src/analysis/leaf_counter.py`) — an estimate, not ground truth;
it under-counts leaves that fully overlap in the 2D projection. Tune
`leaf_min_area` / `leaf_fg_ratio` below if counts look off for your images.

In [ ]:
from src.evaluation.report import plot_pipeline_grid

demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

pipeline_fig_path = plot_pipeline_grid(
    demo_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/pipeline_demo.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)
print('Saved pipeline figure to', pipeline_fig_path)

In [ ]:
import matplotlib.pyplot as plt
import cv2

fig_img = cv2.cvtColor(cv2.imread(str(pipeline_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(demo_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 7c. Test on your own photos

Upload images straight from your device to sanity-check the trained model
on real-world shots outside the dataset's distribution (different framing,
lighting, distance) — including the leaf count. If detections come back
empty, try `conf=0.1` and `green_ratio_threshold=None` first to see raw
model confidence before the post-filters — that's a distribution-shift
finding worth reporting, not necessarily a bug.

In [ ]:
from google.colab import files
from pathlib import Path

upload_dir = Path(f'{PROJECT_ROOT}/data/custom_test')
upload_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # pick photos from your device
custom_images = []
for fname, content in uploaded.items():
    dest = upload_dir / fname
    dest.write_bytes(content)
    custom_images.append(dest)

print(f'Uploaded {len(custom_images)} image(s) to {upload_dir}')

In [ ]:
custom_fig_path = plot_pipeline_grid(
    custom_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/custom_test_pipeline.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)

import matplotlib.pyplot as plt, cv2
fig_img = cv2.cvtColor(cv2.imread(str(custom_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(custom_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 8. Waste detection — second model

A separate, independent model for non-green waste categories (trash,
paper, plastic, soil anomalies, etc.) — deliberately **not** merged into
the vegetation model, so nothing here can change the baseline-vs-masked
ablation results you already have. Default target dataset: [TACO
(Trash Annotations in Context) — YOLO format](https://www.kaggle.com/datasets/vencerlanz09/taco-dataset-yolo-format),
real-scene litter photos with bounding boxes across dozens of fine-grained
categories.

**This dataset's exact category scheme needs verifying after download** —
same discipline as step 3. Run the inspection cell below, look at the
printed class names/ids, then fill in `WASTE_CLASS_MAP` before splitting.
Don't skip straight to training on an unverified guess.

In [ ]:
WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'
import os
os.makedirs(WASTE_RAW_DIR, exist_ok=True)

!kaggle datasets download -d vencerlanz09/taco-dataset-yolo-format -p {WASTE_RAW_DIR} --unzip
!find {WASTE_RAW_DIR} -maxdepth 3 | head -30

In [ ]:
from src.preprocessing.dataset_prep import discover_pairs, inspect_class_distribution, find_class_names

waste_pairs = discover_pairs(Path(WASTE_RAW_DIR))
print(f'Found {len(waste_pairs)} image/label pairs')

waste_class_dist = inspect_class_distribution(waste_pairs)
waste_class_names_raw = find_class_names(Path(WASTE_RAW_DIR))
print('Class distribution:', waste_class_dist)
print('Discovered class names (id -> name, if a classes.txt/yaml was found):')
if waste_class_names_raw:
    for idx, name in enumerate(waste_class_names_raw):
        print(f'  {idx}: {name}  (count={waste_class_dist.get(idx, 0)})')
else:
    print('  No classes.txt/yaml found — cross-reference ids against the Kaggle dataset card manually.')

**Edit this before continuing.** `WASTE_CLASS_MAP` maps *source* class id ->
*output* class id, using the ids/names printed above. Any source id you
leave out of the map is dropped entirely (useful for categories with too
few examples to train on). The placeholder below groups into 5 practical
supercategories — adjust the keys to match what you actually saw printed,
and adjust `WASTE_CLASS_NAMES` to match the values you chose.

In [ ]:
# EDIT THESE TWO based on the class list printed above — this placeholder
# assumes a 0..N raw id scheme and will very likely need remapping.
WASTE_CLASS_NAMES = ['plastic', 'paper', 'metal_glass', 'organic', 'other_rubbish']
WASTE_CLASS_MAP = {
    # source_id: output_id
    0: 0, 1: 0,      # example: plastic bag, plastic bottle -> plastic
    2: 1, 3: 1,       # example: paper, cardboard -> paper
    4: 2, 5: 2,       # example: can, glass jar -> metal_glass
    6: 3,             # example: food waste -> organic
    7: 4,             # example: unlabeled litter -> other_rubbish
}
print('Using WASTE_CLASS_MAP:', WASTE_CLASS_MAP)

In [ ]:
from src.preprocessing.dataset_prep import split_dataset, write_data_yaml

WASTE_DIR = Path(PROJECT_ROOT) / 'data' / 'waste'
split_dataset(waste_pairs, WASTE_DIR, train=0.7, val=0.2, test=0.1, seed=42, class_id_map=WASTE_CLASS_MAP)
waste_yaml = write_data_yaml(WASTE_DIR / 'data.yaml', WASTE_DIR, names=WASTE_CLASS_NAMES)
print('Waste data.yaml:', waste_yaml)

Train the waste model — same wrapper as the vegetation model, just pointed
at a different `data.yaml`. A smaller/faster backbone (`yolov8n`) is a
reasonable default here since this is a secondary model, not the project's
headline ablation; bump to `yolov8s` if you have GPU time to spare.

In [ ]:
waste_cfg = TrainConfig(
    data_yaml=str(waste_yaml),
    model='yolov8n.pt',
    epochs=80,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='waste',
)
waste_model, waste_results = train(waste_cfg)
waste_weights = f'{RUNS_DIR}/waste/weights/best.pt'

## 9. Combined inference — vegetation + waste together

Runs both models independently on the same original image and merges the
two detection sets into one annotated view: green boxes for vegetation,
red boxes for waste categories. This is the "complete scene" artifact —
one image, both models, nothing retrained or merged at the weights level.

In [ ]:
from src.inference.combined_predict import predict_combined_and_save

combined_demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

for img_path in combined_demo_images:
    out_path = f'{PROJECT_ROOT}/reports/combined/{img_path.stem}_combined.jpg'
    predict_combined_and_save(
        img_path, masked_weights, waste_weights, out_path,
        veg_conf=0.25, waste_conf=0.25,
        veg_class_names=['green_vegetation'],
        waste_class_names=WASTE_CLASS_NAMES,
        veg_green_ratio_threshold=0.15,
        mask_config=mask_config,
    )
print('Saved combined annotations to', f'{PROJECT_ROOT}/reports/combined/')

In [ ]:
import matplotlib.pyplot as plt
import cv2

combined_preds = sorted(Path(f'{PROJECT_ROOT}/reports/combined').glob('*_combined.jpg'))
fig, axes = plt.subplots(1, len(combined_preds), figsize=(4 * len(combined_preds), 4))
for ax, p in zip(axes, combined_preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 10. Everything is already on Drive

Since `PROJECT_ROOT` lives under `/content/drive/MyDrive/...`, weights (`runs/*/weights/best.pt`), plots (`reports/`), and the assembled datasets all persist automatically across Colab sessions — no extra export step needed.

In [ ]:
print('Baseline weights:', f'{baseline_run_dir}/weights/best.pt')
print('Masked weights:  ', f'{masked_run_dir}/weights/best.pt')
print('Reports:         ', f'{PROJECT_ROOT}/reports/')
print()
print('Final metrics summary:')
summary
print('Waste weights:     ', f'{RUNS_DIR}/waste/weights/best.pt')

---

# Part 2 — Leaf Detection System for Robotic Collection (Track B)

Everything above (Part 1) is a complete, finished result: HSV green-masking
vs baseline ablation for vegetation detection. **Nothing below touches it.**

This part builds a *separate* system per the robotic-collection spec:
single class `"leaf"`, trained on a **synthetically composited** dataset
(real leaf cutouts + real waste clutter + real harvested backgrounds — no
usable public dataset exists for "fallen leaves mixed with campus waste",
confirmed by search before building this). Green-masking doesn't apply
here — dry leaves aren't green — so this is plain-RGB YOLO training from
scratch, reusing the same `train.py` / `predict.py` / `metrics.py`
machinery from Part 1 wherever it's generic enough to fit.

**Local-disk staging**: cutout extraction, background harvesting, and the
generated synthetic dataset all write to `/content/track_b_work` (the
Colab VM's local disk) instead of Drive. Every read/write against a
Drive-mounted path goes through a network round-trip (Drive FUSE), and
training re-reads every image every epoch — pointing training at Drive
directly turns each epoch into hundreds of network calls instead of local
disk reads. `RUNS_DIR` (checkpoint output) stays on Drive as before —
those writes happen once per epoch, not once per image, so the cost is
negligible and persistence matters there. The synthetic dataset itself is
deterministically regenerable (same seed, same source cutouts/backgrounds)
so losing `/content` on a runtime reset just means re-running B1-B4, not
losing anything irreplaceable — an optional one-shot zip-to-Drive backup
is included at the end of B4 if you'd rather not regenerate.

**Scope for this pass**: Stages 1-4 from the spec (dataset generation → YOLO
leaf detection → image/video inference → counting/tracking) — the part
that's actually gradable. Camera calibration, coordinate transforms, target
selection, the robot command API, and the dashboard are deliberately
deferred until the detector itself is validated; building those against a
model that doesn't exist yet would be scaffolding for its own sake.

## B1. Leaf cutout source — Roboflow leaf-segmentation dataset

We need leaf images with **pixel masks**, not just boxes, so cutouts have
clean silhouettes instead of rectangular seams when pasted. Two small,
free Roboflow Universe projects fit:
[Leaf segmentation (Giovi)](https://universe.roboflow.com/giovi/leaf-segmentation-uxlob),
[Leaf Segmentation (PHD UTM)](https://universe.roboflow.com/phd-utm/leaf-segmentation-rtwov).

**Store your API key as a Colab Secret, never in cell text.** Click the key
icon (🔑) in the left sidebar → *Add new secret* → name it
`ROBOFLOW_API_KEY` → paste your key there → toggle notebook access on.
Secrets never get written into the saved notebook, so they survive
Colab's GitHub auto-save without ever reaching your repo — unlike pasting
a key directly into a cell, which *does* get committed the moment Colab
syncs back to GitHub (this happened once already in this project; the key
was rotated).

Get your project/version numbers from Roboflow's *Download this Dataset* →
format **YOLOv8** → *show download code* on each project page (this export
format gives polygon labels, which is what `cutout_extractor.py`
expects) — copy just the workspace/project/version values, not the whole
snippet with its embedded key.

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

LOCAL_WORK_DIR = '/content/track_b_work'
LEAF_SEG_RAW_DIR = f'{LOCAL_WORK_DIR}/leaf_seg_raw'
import os
os.makedirs(LEAF_SEG_RAW_DIR, exist_ok=True)

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))

# Fill in workspace/project/version from each project's download page —
# just the identifiers, the key itself comes from the secret above.
LEAF_SEG_PROJECTS = [
    # (workspace, project_slug, version_number, subfolder_name)
    ('giovi', 'leaf-segmentation-uxlob', 1, 'giovi'),
    # ('phd-utm', 'leaf-segmentation-rtwov', 1, 'phd_utm'),
]

for workspace, project_slug, version_num, subfolder in LEAF_SEG_PROJECTS:
    project = rf.workspace(workspace).project(project_slug)
    project.version(version_num).download('yolov8', location=f'{LEAF_SEG_RAW_DIR}/{subfolder}')

**If a downloaded project turns out to have plain boxes instead of
polygons** (some Roboflow "segmentation" exports fall back to boxes if the
source annotations weren't actually polygon-drawn), the cutout builder
below will just extract fewer/no instances from it — check the printed
count and swap in `auto_segment_plain_background` from
`cutout_extractor.py` on that subset if needed, rather than assuming
something is broken.

In [ ]:
from pathlib import Path
from src.synthetic.cutout_extractor import extract_cutouts_from_yolo_seg_dataset

LEAF_CUTOUT_DIR = Path(LOCAL_WORK_DIR) / 'leaf_cutouts'
leaf_cutout_paths = extract_cutouts_from_yolo_seg_dataset(Path(LEAF_SEG_RAW_DIR), LEAF_CUTOUT_DIR)
print(f'Extracted {len(leaf_cutout_paths)} leaf cutouts to {LEAF_CUTOUT_DIR}')

## B2. Waste cutout source — reuse the TACO download from Part 1

If you already ran the waste-detection section in Part 1, `WASTE_RAW_DIR`
is already populated — no new download needed. These cutouts are pasted
purely as **unlabeled visual clutter** (the spec is explicit: only `leaf`
is a detection class), so we don't need the class remapping from Part 1
here, just raw bounding-box crops.

In [ ]:
from src.preprocessing.dataset_prep import discover_pairs
from src.synthetic.cutout_extractor import cutout_from_bbox
import cv2

if 'WASTE_RAW_DIR' not in dir():
    WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'  # from Part 1 section 8 — re-download there first if empty

waste_pairs_for_cutouts = discover_pairs(Path(WASTE_RAW_DIR))
WASTE_CUTOUT_DIR = Path(LOCAL_WORK_DIR) / 'waste_cutouts'
WASTE_CUTOUT_DIR.mkdir(parents=True, exist_ok=True)

waste_cutout_paths = []
for img_path, lbl_path in waste_pairs_for_cutouts[:400]:  # cap — we only need clutter variety, not the full dataset
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    h, w = image.shape[:2]
    for i, line in enumerate(lbl_path.read_text().splitlines()):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = line.split()[:5]
        cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
        box = (cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2)
        try:
            cutout = cutout_from_bbox(image, box)
        except ValueError:
            continue
        out_path = WASTE_CUTOUT_DIR / f'{img_path.stem}_{i}.png'
        cv2.imwrite(str(out_path), cutout)
        waste_cutout_paths.append(out_path)

print(f'Extracted {len(waste_cutout_paths)} waste cutouts to {WASTE_CUTOUT_DIR}')

## B3. Background source — harvest from datasets we already have

No public "empty campus ground" dataset exists (checked before building
this). Instead we harvest object-free patches directly from data we
already downloaded: soil-only regions of the Part-1 crop/weed dataset
(inverted green mask, away from any labeled box) and non-litter regions of
the TACO images. Both are real photos of real outdoor ground.

In [ ]:
from src.synthetic.background_harvester import harvest_from_green_dataset, harvest_from_boxed_dataset
from src.preprocessing.hsv_mask import MaskConfig

BACKGROUND_DIR = Path(LOCAL_WORK_DIR) / 'backgrounds'

green_ds_pairs = discover_pairs(Path(RAW_DATA_DIR))  # Part 1's crop/weed raw data
bg_from_green = harvest_from_green_dataset(
    green_ds_pairs, BACKGROUND_DIR / 'from_crop_weed', patch_size=256, patches_per_image=2,
    mask_config=mask_config if 'mask_config' in dir() else MaskConfig(),
)

bg_from_waste = harvest_from_boxed_dataset(
    waste_pairs_for_cutouts, BACKGROUND_DIR / 'from_taco', patch_size=256, patches_per_image=1,
)

background_paths = bg_from_green + bg_from_waste
print(f'Harvested {len(background_paths)} background patches ({len(bg_from_green)} soil, {len(bg_from_waste)} scene)')

## B4. Generate the synthetic training set

This is the "Controlled Synthetic Generation" step — paste cutouts onto
backgrounds with randomized geometry across easy/medium/hard difficulty
tiers. Bounding boxes come out of the paste operation automatically, so
there's no manual annotation step at all. **Look at the sanity-check grid
below before committing GPU time to training** — if leaves look obviously
fake (wrong scale, floating without shadow, always centered), tune
`n_leaves_range` / `DIFFICULTY_PRESETS` in `src/synthetic/compositor.py`
first.

In [ ]:
from src.synthetic.generate_dataset import generate_synthetic_dataset

LEAF_DATASET_DIR = Path(LOCAL_WORK_DIR) / 'leaf_synthetic'  # local — read every training epoch, must not be on Drive
generate_synthetic_dataset(
    leaf_cutout_paths, background_paths, LEAF_DATASET_DIR,
    waste_cutout_paths=waste_cutout_paths,
    n_train=900, n_val=150, n_test=150,
    n_leaves_range=(2, 8), canvas_size=(640, 640), seed=42,
)
leaf_data_yaml = LEAF_DATASET_DIR / 'data.yaml'
print('Leaf data.yaml:', leaf_data_yaml)

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample_imgs = sorted((LEAF_DATASET_DIR / 'images' / 'train').iterdir())[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    lbl_path = LEAF_DATASET_DIR / 'labels' / 'train' / img_path.with_suffix('.txt').name
    for line in lbl_path.read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(image)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis('off')
fig.tight_layout()
plt.show()

**Optional** — back up the generated dataset to Drive as a single zip
(one bulk write, not 1200+ small ones, so it's fast). Skip this if you're
fine regenerating from B1-B4 next session — it's deterministic (same seed,
same source cutouts) as long as the cutout/background pools haven't
changed.

In [ ]:
import shutil

BACKUP_TO_DRIVE = False  # flip to True if you want a persisted copy

if BACKUP_TO_DRIVE:
    zip_base = f'{PROJECT_ROOT}/data/leaf_synthetic_backup'
    shutil.make_archive(zip_base, 'zip', LEAF_DATASET_DIR)
    print('Backed up to', zip_base + '.zip')
else:
    print('Skipped — regenerate via B1-B4 next session if needed.')

## B5. Train the leaf detector

Same wrapper as Part 1, pointed at the synthetic dataset, single class
`leaf`. `yolov8s` is a reasonable default; drop to `yolov8n` if Colab's
free-tier GPU queue is tight on time.

In [ ]:
leaf_cfg = TrainConfig(
    data_yaml=str(leaf_data_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name='leaf',
)
leaf_model, leaf_results = train(leaf_cfg)
leaf_weights = f'{RUNS_DIR}/leaf/weights/best.pt'

## B6. Evaluate

In [ ]:
leaf_run_dir = f'{RUNS_DIR}/leaf'
leaf_df = load_results_csv(leaf_run_dir)
print(summarize_final_metrics(leaf_df))
plot_loss_curves(leaf_df, f'{PROJECT_ROOT}/reports/leaf_loss.png', 'Leaf Detector — Loss')
plot_map_curves(leaf_df, f'{PROJECT_ROOT}/reports/leaf_map.png', 'Leaf Detector — mAP')

## B7. Image inference — spec-shaped JSON output

This is the interface contract a future robot-side component would
consume (spec Section 20). No robot code exists yet, but the output shape
already matches what one would need.

In [ ]:
import json
from src.inference.track import predict_image_response

test_leaf_images = list((LEAF_DATASET_DIR / 'images' / 'test').iterdir())[:3]
for img_path in test_leaf_images:
    response = predict_image_response(leaf_weights, img_path, conf=0.25, class_names=['leaf'])
    print(img_path.name, '->', json.dumps(response, indent=2))

## B8. Video tracking demo

Upload a short clip if you have one (leaves/ground, a few seconds is
enough). If not, the cell below builds a small synthetic demo clip by
panning the compositor across a few frames, purely so the tracking API can
be exercised end-to-end without needing real footage yet.

In [ ]:
from google.colab import files

UPLOAD_REAL_VIDEO = False  # flip to True if you're uploading your own clip

if UPLOAD_REAL_VIDEO:
    uploaded = files.upload()
    video_path = Path(PROJECT_ROOT) / 'data' / list(uploaded.keys())[0]
    video_path.write_bytes(list(uploaded.values())[0])
else:
    from src.synthetic.compositor import compose_scene
    video_path = Path(PROJECT_ROOT) / 'data' / 'synthetic_demo_clip.mp4'
    bg = cv2.imread(str(background_paths[0]))
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*'mp4v'), 5, (bg.shape[1], bg.shape[0]))
    for i in range(20):
        demo_cutouts = [cv2.imread(str(p), cv2.IMREAD_UNCHANGED) for p in leaf_cutout_paths[:5]]
        result = compose_scene(bg, demo_cutouts, n_leaves=(3, 3), difficulty='easy', seed=i)
        writer.write(result.image)
    writer.release()
    print('Synthetic demo clip (not real footage) written to', video_path)

In [ ]:
from src.inference.track import track_video_to_json

track_out = track_video_to_json(leaf_weights, str(video_path), f'{PROJECT_ROOT}/reports/leaf_tracks.json', conf=0.25, class_names=['leaf'])
print('Per-frame tracking results saved to', track_out)

import json
frames = json.loads(track_out.read_text())
print(f'{len(frames)} frames tracked. Sample frame:')
print(json.dumps(frames[len(frames)//2], indent=2))

## B9. Everything is on Drive

Same pattern as Part 1 — everything under `PROJECT_ROOT` persists to
Google Drive automatically:

- `data/leaf_cutouts/`, `data/waste_cutouts/`, `data/backgrounds/` — the
  reusable source material for regenerating or extending the synthetic set
- `data/leaf_synthetic/` — the generated training set itself
- `runs/leaf/weights/best.pt` — the trained leaf detector
- `reports/leaf_tracks.json` — the tracking demo output

**Deferred to a later pass** (per the "CV core first" scope decision):
camera calibration, coordinate transforms, target selection, the mock
robot command API, the state machine, and the dashboard. Nothing here
blocks adding them once the detector's real-world accuracy is validated —
that was the whole point of building this before the robot-facing layers.